# MedVis Exercise Sheet 1 — Student Solution

This notebook solves **Task 4 (Read DICOM Data)** and **Task 5 (Visualization of 3D Medical Image Data)** using the provided additional material notebook as a base.

## Preparation
Install and import all required libraries.

In [ ]:
!pip install scipy
!pip install pydicom

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from pydicom import dcmread

## Helper Function: Read DICOM Volume

This function reads all DICOM slices from a folder and stacks them into a 3D numpy array.
Each slice file is one image in the z-direction.

In [ ]:
def read_dcm_volume(name):
    volume_img = []
    for file in sorted(os.listdir(name)):
        if file.endswith('.dcm'):
            dcm_slice = dcmread(os.path.join(name, file))
            volume_img.append(dcm_slice.pixel_array)
    return np.transpose(np.array(volume_img), (1, 2, 0))

## Task 4: Read DICOM Data

Read one DICOM slice and print the relevant imaging parameters:
- Imaging Modality
- Slice Thickness in z-direction
- Pixel Spacing
- Grid Size
- Smallest Image Pixel Value
- Largest Image Pixel Value

In [ ]:
# Read one DICOM slice using dcmread()
# Make sure 'dataset1/' contains the brain_001.dcm ... brain_020.dcm files
dcm_slice = dcmread('dataset1/brain_001.dcm')

# Print the full DICOM metadata
print(dcm_slice)

In [ ]:
# Extract and display the specific parameters required by the exercise
print('===== Task 4: DICOM Parameters =====')
print(f'Imaging Modality:          {dcm_slice.Modality}')
print(f'Slice Thickness (z):       {dcm_slice.SliceThickness} mm')
print(f'Pixel Spacing (x, y):      {dcm_slice.PixelSpacing[0]} x {dcm_slice.PixelSpacing[1]} mm')
print(f'Grid Size (Rows x Cols):   {dcm_slice.Rows} x {dcm_slice.Columns} pixels')
print(f'Smallest Image Pixel Value:{dcm_slice.SmallestImagePixelValue}')
print(f'Largest Image Pixel Value: {dcm_slice.LargestImagePixelValue}')

### Task 4 Results

| Parameter | Value |
|---|---|
| Imaging Modality | MR (Magnetic Resonance Imaging) |
| Slice Thickness (z) | 5.0 mm |
| Pixel Spacing | 0.859375 x 0.859375 mm |
| Grid Size | 256 x 256 pixels |
| Smallest Image Pixel Value | 0 |
| Largest Image Pixel Value | 884 |

## Task 5: Visualization of 3D Medical Image Data

Load the full 3D volume and display one **axial**, one **sagittal**, and one **coronal** slice.

The volume size is [256, 256, 20] — meaning:
- x (coronal): 256 slices
- y (sagittal): 256 slices
- z (axial): 20 slices

In [ ]:
# Load the full 3D volume from the dataset folder
dcm_volume = read_dcm_volume('dataset1')
print(f'Volume shape: {dcm_volume.shape}')  # Expected: (256, 256, 20)

In [ ]:
# Choose middle indices for each orientation
axial_idx   = 10  # z-direction: middle of 0-19
sagittal_idx = 128  # y-direction: middle of 0-255
coronal_idx  = 128  # x-direction: middle of 0-255

# Extract slices
# Axial: fix z, take all x and y
axial_slice   = dcm_volume[:, :, axial_idx]
# Sagittal: fix y, take all x and z
sagittal_slice = dcm_volume[:, sagittal_idx, :]
# Coronal: fix x, take all y and z
coronal_slice  = dcm_volume[coronal_idx, :, :]

print(f'Axial slice shape:    {axial_slice.shape}')    # (256, 256)
print(f'Sagittal slice shape: {sagittal_slice.shape}') # (256, 20) — anisotropic!
print(f'Coronal slice shape:  {coronal_slice.shape}')  # (256, 20) — anisotropic!

In [ ]:
# Display the three slices WITHOUT correcting for anisotropy
# Notice how the sagittal and coronal slices look squashed/stretched

fig = plt.figure(figsize=(12, 4))
fig.suptitle('Task 5: Axial, Sagittal, and Coronal Slices (Anisotropic)', fontsize=14)

# Axial
axial = plt.subplot(1, 3, 1)
plt.title(f'Axial (z={axial_idx})')
plt.imshow(axial_slice, cmap='gray')
plt.axis('off')

# Sagittal
sagittal = plt.subplot(1, 3, 2)
plt.title(f'Sagittal (y={sagittal_idx})')
plt.imshow(sagittal_slice, cmap='gray')
plt.axis('off')

# Coronal
coronal = plt.subplot(1, 3, 3)
plt.title(f'Coronal (x={coronal_idx})')
plt.imshow(coronal_slice, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()

### Anisotropy Problem — Explanation

As you can see, the **sagittal and coronal slices** look very squashed compared to the axial slice.

This is because:
- In-plane (x, y) resolution: **0.859 mm per pixel** → 256 pixels = ~220 mm
- Slice thickness (z): **5.0 mm per slice** → 20 slices = 100 mm

So the z-direction is much coarser. When we cut sagittally or coronally, we only have 20 data points in z, making the image look squashed.

**Solution:** Resample (interpolate) the volume to make the voxels isotropic — i.e., equal size in all three directions. This is shown in the next cell.

In [ ]:
# Fix the anisotropy using zoom interpolation
# Pixel spacing in x,y = 0.859375 mm, slice thickness in z = 5.0 mm
# Zoom factor in z = 5.0 / 0.859375 ≈ 5.82

pixel_spacing = 0.859375  # mm
slice_thickness = 5.0     # mm
zoom_factor_z = slice_thickness / pixel_spacing

# Resample volume: keep x and y the same, stretch z
dcm_volume_resampled = ndimage.zoom(dcm_volume, (1, 1, zoom_factor_z))
print(f'Resampled volume shape: {dcm_volume_resampled.shape}')

In [ ]:
# Extract new slices from the resampled (isotropic) volume
z_mid = dcm_volume_resampled.shape[2] // 2

axial_iso   = dcm_volume_resampled[:, :, z_mid]
sagittal_iso = dcm_volume_resampled[:, 128, :]
coronal_iso  = dcm_volume_resampled[128, :, :]

# Display corrected slices
fig = plt.figure(figsize=(12, 4))
fig.suptitle('Task 5: Slices After Anisotropy Correction (Isotropic Resampling)', fontsize=14)

axial = plt.subplot(1, 3, 1)
plt.title('Axial (corrected)')
plt.imshow(axial_iso, cmap='gray')
plt.axis('off')

sagittal = plt.subplot(1, 3, 2)
plt.title('Sagittal (corrected)')
plt.imshow(sagittal_iso, cmap='gray')
plt.axis('off')

coronal = plt.subplot(1, 3, 3)
plt.title('Coronal (corrected)')
plt.imshow(coronal_iso, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()